# Investigating Auto Theft Anomalies

## Background
In our previous exploratory data analysis of Toronto's crime data, one specific neighborhood (**West Humber-Clairville**) stood out as an extreme outlier in auto theft rates. While auto theft is a city-wide issue, the concentration of incidents in this particular area is disproportionately high and deviates significantly from standard city patterns.


Instead of assuming that the high theft rate is arbitrary, we will evaluate whether proximity to key logistical hubs creates a functional ecosystem that facilitates these crimes. We will analyze the spatial relationship between theft locations and:
* **Highways junctions:** quick getaway.
* **Industrial warehouses:** potential temporary storage or "cooling off" locations.
* **Airport Parking:** vulnerable, large-scale, and unmonitored vehicle storage.
* **Car dealerships and repair shops:** high-value targets and potential dismantling sites.

## Methodology
By applying geospatial analysis (Haversine distance matrices) and statistical hypothesis testing (t-test), we aim to mathematically validate whether the infrastructure of West Humber significantly differs from the rest of Toronto in the context of auto theft logistics.

## 1. Data Ingestion & Preprocessing

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Dim_Location_Demographics.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 15 columns):
 #   Column                                                                                                                            Non-Null Count  Dtype  
---  ------                                                                                                                            --------------  -----  
 0   NEIGHBOURHOOD_NAME                                                                                                                158 non-null    object 
 1   HOOD_158                                                                                                                          158 non-null    int64  
 2   Total - Visible minority for the population in private households - 25% sample data                                               158 non-null    int64  
 3   Total - Private households by tenure - 25% sample data                                         

In [4]:
df.head(5)

,NEIGHBOURHOOD_NAME,HOOD_158,Total - Visible minority for the population in private households - 25% sample data,Total - Private households by tenure - 25% sample data,Owner,Renter,Total - Immigrant status and period of immigration for the population in private households - 25% sample data,Non-immigrants,"Prevalence of low income based on the Low-income measure, after tax (LIM-AT) (%)",Unemployment rate,Median total income of household in 2020 ($),Average total income of household in 2020 ($),Employment rate,"Total - Highest certificate, diploma or degree for the population aged 15 years and over in private households - 25% sample data","No certificate, diploma or degree"
0,West Humber-Clairville,1,33300,10700,6995,3705,33300,11805,8.7,14.3,92000,104500,54.4,29000,5305
1,Mount Olive-Silverstone-Jamestown,2,31345,9740,4485,5260,31345,9620,15.6,17.7,76500,86200,46.1,25655,7130
2,Thistletown-Beaumond Heights,3,9850,3245,2025,1220,9850,4055,11.8,16.6,86000,101300,49.9,8350,1860
3,Rexdale-Kipling,4,10375,3945,1990,1955,10375,5080,13.5,13.9,77000,90000,51.8,8805,1870
4,Elms-Old Rexdale,5,9355,3195,1760,1430,9355,4515,12.4,18.5,82000,94600,46.5,7745,1755


In [5]:
df1 = pd.read_csv('MCI.csv')


In [6]:
df1.head(5)

,OBJECTID_1,NEIGHBOURHOOD_NAME,HOOD_158,ASSAULT_2014,ASSAULT_2015,ASSAULT_2016,ASSAULT_2017,ASSAULT_2018,ASSAULT_2019,ASSAULT_2020,...,THEFTOVER_RATE_2019,THEFTOVER_RATE_2020,THEFTOVER_RATE_2021,THEFTOVER_RATE_2022,THEFTOVER_RATE_2023,THEFTOVER_RATE_2024,THEFTOVER_RATE_2025,POPULATION_2025,Shape__Area,Shape__Length
0,1,South Eglinton-Davisville,174,55,56,66,73,74,62,74,...,13.097577,16.740604,24.216984,11.665436,29.499613,21.188684,20.759809,28902,9.443090e+05,4005.549887
1,2,North Toronto,173,53,57,47,61,66,84,80,...,26.750484,43.862396,11.872959,22.540291,36.945164,30.007502,24.293072,20582,4.020308e+05,2543.945884
2,3,Dovercourt Village,172,62,65,92,105,106,113,91,...,29.775198,22.602276,15.283509,30.555344,22.610792,52.426601,22.850180,13129,1.503002e+06,4965.496448
3,4,Junction-Wallace Emerson,171,164,159,171,161,163,186,171,...,15.985932,36.070698,32.293224,31.383625,33.834587,47.279606,32.400906,27777,2.222867e+06,7435.192384
4,5,Yonge-Bay Corridor,170,387,521,481,602,576,660,377,...,478.723419,258.569733,187.630798,351.821198,337.443207,296.040466,210.071426,16661,1.118725e+06,4811.107669


In [7]:
df2 = pd.read_csv('MCI.csv')

In [8]:
df2.head(5)

,OBJECTID_1,NEIGHBOURHOOD_NAME,HOOD_158,ASSAULT_2014,ASSAULT_2015,ASSAULT_2016,ASSAULT_2017,ASSAULT_2018,ASSAULT_2019,ASSAULT_2020,...,THEFTOVER_RATE_2019,THEFTOVER_RATE_2020,THEFTOVER_RATE_2021,THEFTOVER_RATE_2022,THEFTOVER_RATE_2023,THEFTOVER_RATE_2024,THEFTOVER_RATE_2025,POPULATION_2025,Shape__Area,Shape__Length
0,1,South Eglinton-Davisville,174,55,56,66,73,74,62,74,...,13.097577,16.740604,24.216984,11.665436,29.499613,21.188684,20.759809,28902,9.443090e+05,4005.549887
1,2,North Toronto,173,53,57,47,61,66,84,80,...,26.750484,43.862396,11.872959,22.540291,36.945164,30.007502,24.293072,20582,4.020308e+05,2543.945884
2,3,Dovercourt Village,172,62,65,92,105,106,113,91,...,29.775198,22.602276,15.283509,30.555344,22.610792,52.426601,22.850180,13129,1.503002e+06,4965.496448
3,4,Junction-Wallace Emerson,171,164,159,171,161,163,186,171,...,15.985932,36.070698,32.293224,31.383625,33.834587,47.279606,32.400906,27777,2.222867e+06,7435.192384
4,5,Yonge-Bay Corridor,170,387,521,481,602,576,660,377,...,478.723419,258.569733,187.630798,351.821198,337.443207,296.040466,210.071426,16661,1.118725e+06,4811.107669


In [9]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Columns: 222 entries, OBJECTID_1 to Shape__Length
dtypes: float64(110), int64(111), object(1)
memory usage: 274.2+ KB


In [10]:
cols_to_sum = df2.loc[:, 'AUTOTHEFT_2014':'AUTOTHEFT_2025'].columns

yearly_thefts_by_hood = df2.groupby('NEIGHBOURHOOD_NAME')[cols_to_sum].sum()

yearly_thefts_by_hood['total_autotheft'] = yearly_thefts_by_hood.sum(axis=1)
display(yearly_thefts_by_hood.sort_values(by='total_autotheft', ascending=False))

,AUTOTHEFT_2014,AUTOTHEFT_2015,AUTOTHEFT_2016,AUTOTHEFT_2017,AUTOTHEFT_2018,AUTOTHEFT_2019,AUTOTHEFT_2020,AUTOTHEFT_2021,AUTOTHEFT_2022,AUTOTHEFT_2023,AUTOTHEFT_2024,AUTOTHEFT_2025,total_autotheft
NEIGHBOURHOOD_NAME,,,,,,,,,,,,,
West Humber-Clairville,311,270,322,322,496,485,398,500,777,843,661,494,5879
York University Heights,112,107,110,94,97,152,190,173,238,268,227,213,1981
Etobicoke City Centre,67,82,103,72,118,119,103,120,188,296,270,186,1724
Milliken,37,45,35,59,87,83,70,77,108,319,200,160,1280
Humber Summit,68,42,62,80,110,134,110,139,171,136,117,88,1257
...,...,...,...,...,...,...,...,...,...,...,...,...,...
Broadview North,2,4,8,8,5,3,8,12,15,16,15,24,120
North Toronto,8,2,2,2,3,5,14,15,10,20,10,14,105
Guildwood,2,1,4,4,2,3,12,13,9,23,17,8,98


In [11]:
df3 = pd.read_excel('profiles.xlsx')

In [12]:
df3.head(15)

,Neighbourhood Name,West Humber-Clairville,Mount Olive-Silverstone-Jamestown,Thistletown-Beaumond Heights,Rexdale-Kipling,Elms-Old Rexdale,Kingsview Village-The Westway,Willowridge-Martingrove-Richview,Humber Heights-Westmount,Edenbridge-Humber Valley,...,Harbourfront-CityPlace,St Lawrence-East Bayfront-The Islands,Church-Wellesley,Downtown Yonge East,Bay-Cloverhill,Yonge-Bay Corridor,Junction-Wallace Emerson,Dovercourt Village,North Toronto,South Eglinton-Davisville
0,Neighbourhood Number,1,2,3,4,5,6,7,8,9,...,165,166,167,168,169,170,171,172,173,174
1,TSNS 2020 Designation,Not an NIA or Emerging Neighbourhood,Neighbourhood Improvement Area,Neighbourhood Improvement Area,Not an NIA or Emerging Neighbourhood,Neighbourhood Improvement Area,Neighbourhood Improvement Area,Not an NIA or Emerging Neighbourhood,Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,...,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood,Not an NIA or Emerging Neighbourhood
2,Total - Age groups of the population - 25% sam...,33300,31345,9850,10375,9355,22005,22445,10005,15190,...,28135,31285,22320,17700,16670,12645,23180,12380,15885,22735
3,0 to 14 years,4295,5690,1495,1575,1610,3915,3500,1370,2070,...,2065,2285,895,1055,745,970,3075,1365,1315,2190
4,0 to 4 years,1460,1650,505,505,440,1245,1065,395,520,...,1030,1045,495,480,370,500,1135,445,535,910
5,5 to 9 years,1345,1860,540,615,480,1325,1190,430,740,...,655,690,230,325,255,270,1020,430,390,670
6,10 to 14 years,1485,2175,455,455,685,1350,1240,540,815,...,385,550,175,245,120,200,925,490,390,610
7,15 to 64 years,23640,21490,6615,6950,6355,14385,13865,6245,9650,...,24540,25015,19315,14695,13815,10820,17200,9040,12780,17495
8,15 to 19 years,1860,2280,570,515,635,1245,1175,525,885,...,365,600,465,390,535,340,750,460,465,575
9,20 to 24 years,3175,2675,745,715,685,1605,1330,580,780,...,2250,1990,3010,2340,3810,1825,1185,640,960,1115


In [13]:
df_temp = df3.transpose().drop(columns = 0)
df_temp.columns = df_temp.loc['Neighbourhood Name']
df_final = df_temp.drop(index='Neighbourhood Name')
display(df_final)

Neighbourhood Name,TSNS 2020 Designation,Total - Age groups of the population - 25% sample data,0 to 14 years,0 to 4 years,5 to 9 years,10 to 14 years,15 to 64 years,15 to 19 years,20 to 24 years,25 to 29 years,...,Between 9 a.m. and 11:59 a.m.,Between 12 p.m. and 4:59 a.m.,Total - Eligibility for instruction in the minority official language for the population in private households born in 2003 or later - 25% sample data,Children eligible for instruction in the minority official language,Children not eligible for instruction in the minority official language,"Total - Eligibility and instruction in the minority official language, for the population in private households born between 2003 and 2015 (inclusive) - 25% sample data",Children eligible for instruction in the minority official language,Eligible children who have been instructed in the minority official language at the primary or secondary level in Canada,Eligible children who have not been instructed in the minority official language at the primary or secondary level in Canada,Children not eligible for instruction in the minority official language
West Humber-Clairville,Not an NIA or Emerging Neighbourhood,33300,4295,1460,1345,1485,23640,1860,3175,3585,...,1665,2935,5430,410,5020,3875,335,255,75,3540
Mount Olive-Silverstone-Jamestown,Neighbourhood Improvement Area,31345,5690,1650,1860,2175,21490,2280,2675,2550,...,1145,2965,7285,510,6780,5540,395,245,145,5145
Thistletown-Beaumond Heights,Neighbourhood Improvement Area,9850,1495,505,540,455,6615,570,745,880,...,395,635,1860,180,1685,1325,120,75,45,1205
Rexdale-Kipling,Not an NIA or Emerging Neighbourhood,10375,1575,505,615,455,6950,515,715,785,...,425,775,1910,135,1770,1370,90,75,25,1275
Elms-Old Rexdale,Neighbourhood Improvement Area,9355,1610,440,480,685,6355,635,685,730,...,355,675,2015,95,1920,1520,70,60,10,1445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Yonge-Bay Corridor,Not an NIA or Emerging Neighbourhood,12645,970,500,270,200,10820,340,1825,2750,...,695,330,1075,50,1020,555,30,10,20,520
Junction-Wallace Emerson,Not an NIA or Emerging Neighbourhood,23180,3075,1135,1020,925,17200,750,1185,2190,...,1185,1105,3580,410,3170,2375,305,190,115,2075
Dovercourt Village,Not an NIA or Emerging Neighbourhood,12380,1365,445,430,490,9040,460,640,1270,...,775,570,1665,175,1490,1190,130,95,35,1060
North Toronto,Not an NIA or Emerging Neighbourhood,15885,1315,535,390,390,12780,465,960,2325,...,970,550,1620,145,1470,1050,95,65,30,955


In [14]:
#df4 = pd.read_csv('info_n.csv')

In [15]:
#df4.head(4)

In [16]:
import geopandas as gpd

In [17]:
geo_data = gpd.read_file('info_n.geojson')

In [18]:
geo_data.head(5)

,_id,AREA_ID,AREA_ATTR_ID,PARENT_AREA_ID,AREA_SHORT_CODE,AREA_LONG_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,OBJECTID,geometry
0,1,2502366,26022881,None,174,174,South Eglinton-Davisville,South Eglinton-Davisville (174),Not an NIA or Emerging Neighbourhood,NA,17824737.0,"MULTIPOLYGON (((-79.38635 43.69783, -79.38623 ..."
1,2,2502365,26022880,None,173,173,North Toronto,North Toronto (173),Not an NIA or Emerging Neighbourhood,NA,17824753.0,"MULTIPOLYGON (((-79.39744 43.70693, -79.39837 ..."
2,3,2502364,26022879,None,172,172,Dovercourt Village,Dovercourt Village (172),Not an NIA or Emerging Neighbourhood,NA,17824769.0,"MULTIPOLYGON (((-79.43411 43.66015, -79.43537 ..."
3,4,2502363,26022878,None,171,171,Junction-Wallace Emerson,Junction-Wallace Emerson (171),Not an NIA or Emerging Neighbourhood,NA,17824785.0,"MULTIPOLYGON (((-79.4387 43.66766, -79.43841 4..."
4,5,2502362,26022877,None,170,170,Yonge-Bay Corridor,Yonge-Bay Corridor (170),Not an NIA or Emerging Neighbourhood,NA,17824801.0,"MULTIPOLYGON (((-79.38404 43.64497, -79.38502 ..."


In [19]:
geo_data['centroid'] = geo_data['geometry'].centroid

/tmp/ipykernel_209/236377888.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  geo_data['centroid'] = geo_data['geometry'].centroid


In [20]:
geo_data.head(5)

,_id,AREA_ID,AREA_ATTR_ID,PARENT_AREA_ID,AREA_SHORT_CODE,AREA_LONG_CODE,AREA_NAME,AREA_DESC,CLASSIFICATION,CLASSIFICATION_CODE,OBJECTID,geometry,centroid
0,1,2502366,26022881,None,174,174,South Eglinton-Davisville,South Eglinton-Davisville (174),Not an NIA or Emerging Neighbourhood,NA,17824737.0,"MULTIPOLYGON (((-79.38635 43.69783, -79.38623 ...",POINT (-79.39262 43.70193)
1,2,2502365,26022880,None,173,173,North Toronto,North Toronto (173),Not an NIA or Emerging Neighbourhood,NA,17824753.0,"MULTIPOLYGON (((-79.39744 43.70693, -79.39837 ...",POINT (-79.39507 43.71025)
2,3,2502364,26022879,None,172,172,Dovercourt Village,Dovercourt Village (172),Not an NIA or Emerging Neighbourhood,NA,17824769.0,"MULTIPOLYGON (((-79.43411 43.66015, -79.43537 ...",POINT (-79.4288 43.66623)
3,4,2502363,26022878,None,171,171,Junction-Wallace Emerson,Junction-Wallace Emerson (171),Not an NIA or Emerging Neighbourhood,NA,17824785.0,"MULTIPOLYGON (((-79.4387 43.66766, -79.43841 4...",POINT (-79.44512 43.6653)
4,5,2502362,26022877,None,170,170,Yonge-Bay Corridor,Yonge-Bay Corridor (170),Not an NIA or Emerging Neighbourhood,NA,17824801.0,"MULTIPOLYGON (((-79.38404 43.64497, -79.38502 ...",POINT (-79.38373 43.65295)


In [21]:
geo_data['centroid_x'] = geo_data['centroid'].x
geo_data['centroid_y'] = geo_data['centroid'].y

In [22]:
autotheft = pd.read_csv('autotheft.csv')

In [23]:
autotheft.head(5)

,OBJECTID,EVENT_UNIQUE_ID,REPORT_DATE,OCC_DATE,REPORT_YEAR,REPORT_MONTH,REPORT_DAY,REPORT_DOY,REPORT_DOW,REPORT_HOUR,...,OFFENCE,CSI_CATEGORY,HOOD_158,NEIGHBOURHOOD_158,HOOD_140,NEIGHBOURHOOD_140,LONG_WGS84,LAT_WGS84,x,y
0,1,GO-20141262837,1/1/2014 5:00:00 AM,12/25/2013 5:00:00 AM,2014,January,1,1,Wednesday,15,...,Theft Of Motor Vehicle,Auto Theft,159,Etobicoke City Centre (159),014,Islington-City Centre West (14),-79.529692,43.618988,-8.853205e+06,5.406668e+06
1,2,GO-20141263217,1/1/2014 5:00:00 AM,12/31/2013 5:00:00 AM,2014,January,1,1,Wednesday,16,...,Theft Of Motor Vehicle,Auto Theft,043,Victoria Village (43),043,Victoria Village (43),-79.306754,43.734654,-8.828387e+06,5.424471e+06
2,3,GO-20141262914,1/1/2014 5:00:00 AM,1/1/2014 5:00:00 AM,2014,January,1,1,Wednesday,15,...,Theft Of Motor Vehicle,Auto Theft,123,Cliffcrest (123),123,Cliffcrest (123),-79.236119,43.721827,-8.820524e+06,5.422495e+06
3,4,GO-20141266240,1/2/2014 5:00:00 AM,1/2/2014 5:00:00 AM,2014,January,2,2,Thursday,9,...,Theft Of Motor Vehicle,Auto Theft,060,Woodbine-Lumsden (60),060,Woodbine-Lumsden (60),-79.313796,43.688101,-8.829171e+06,5.417301e+06
4,5,GO-20141266097,1/2/2014 5:00:00 AM,1/2/2014 5:00:00 AM,2014,January,2,2,Thursday,8,...,Theft Of Motor Vehicle,Auto Theft,129,Agincourt North (129),129,Agincourt North (129),-79.273925,43.813557,-8.824733e+06,5.436635e+06


## 2. Querying Open Location Data

In this section, we retrieve external spatial data detailing the locations of key logistical and automotive hubs. This open data serves as the foundation for building our distance matrices.

In [24]:
import requests as rq

In [25]:
url = 'http://overpass-api.de/api/interpreter'

In [26]:
query = '''
[out:json];
area["name"="Toronto"]->.searchArea;
(
  node["shop"="car"](area.searchArea);
  way["shop"="car"](area.searchArea);
);
out center;
'''

In [27]:
response = rq.post(url, data={'data': query})

data_dict = response.json()

print(data_dict.keys())

dict_keys(['version', 'generator', 'osm3s', 'elements'])


In [28]:
dealers_list = []
for element in data_dict['elements']:
  if element['type'] == 'node':
    latitude = element['lat']
    longtitude = element['lon']
  elif element['type'] == 'way':
    latitude = element['center']['lat']
    longtitude = element['center']['lon']
  coordinates = {'Dealer_Lat': latitude, 'Dealer_Lon': longtitude}
  dealers_list.append(coordinates)

In [29]:
print(dealers_list)

[{'Dealer_Lat': 43.6933513, 'Dealer_Lon': -79.3158558}, {'Dealer_Lat': 43.7254707, 'Dealer_Lon': -79.3123292}, {'Dealer_Lat': 43.7063194, 'Dealer_Lon': -79.5311875}, {'Dealer_Lat': 43.6853147, 'Dealer_Lon': -79.3467768}, {'Dealer_Lat': 43.7691719, 'Dealer_Lon': -79.4674438}, {'Dealer_Lat': 43.6534318, 'Dealer_Lon': -79.3643267}, {'Dealer_Lat': 43.7184947, 'Dealer_Lon': -79.3493157}, {'Dealer_Lat': 43.7502264, 'Dealer_Lon': -79.2736333}, {'Dealer_Lat': 43.6725731, 'Dealer_Lon': -79.3953395}, {'Dealer_Lat': 43.775087, 'Dealer_Lon': -79.492565}, {'Dealer_Lat': 43.6575771, 'Dealer_Lon': -79.3527669}, {'Dealer_Lat': 43.659442, 'Dealer_Lon': -79.3558386}, {'Dealer_Lat': 43.7647566, 'Dealer_Lon': -79.4900717}, {'Dealer_Lat': 43.804615, 'Dealer_Lon': -79.298032}, {'Dealer_Lat': 43.800464, 'Dealer_Lon': -79.189521}, {'Dealer_Lat': 43.6188062, 'Dealer_Lon': -79.527624}, {'Dealer_Lat': 43.6569283, 'Dealer_Lon': -79.3486178}, {'Dealer_Lat': 43.770898, 'Dealer_Lon': -79.468399}, {'Dealer_Lat': 43.6

In [30]:
df_dealers = pd.DataFrame(dealers_list)
display(df_dealers.head())

,Dealer_Lat,Dealer_Lon
0,43.693351,-79.315856
1,43.725471,-79.312329
2,43.706319,-79.531188
3,43.685315,-79.346777
4,43.769172,-79.467444


In [31]:
query_2 = '''
[out:json];
area["name"="Toronto"]->.searchArea;
(
  node["highway"="motorway_junction"](area.searchArea);
);
out center;
'''

In [32]:
response = rq.post(url, data={'data': query_2})

data_dict_2 = response.json()

In [33]:
junctions_list = []
for element in data_dict_2['elements']:
  if element['type'] == 'node':
    latitude = element['lat']
    longtitude = element['lon']
  elif element['type'] == 'way':
    latitude = element['center']['lat']
    longtitude = element['center']['lon']
  coordinates = {'Junction_Lat': latitude, 'Junction_Lon': longtitude}
  junctions_list.append(coordinates)

In [34]:
print(junctions_list)

[{'Junction_Lat': 43.7555309, 'Junction_Lon': -79.3346901}, {'Junction_Lat': 43.7255298, 'Junction_Lon': -79.3305335}, {'Junction_Lat': 43.7689419, 'Junction_Lon': -79.3343753}, {'Junction_Lat': 43.7675254, 'Junction_Lon': -79.3343615}, {'Junction_Lat': 43.7666169, 'Junction_Lon': -79.3536635}, {'Junction_Lat': 43.7682983, 'Junction_Lon': -79.3273776}, {'Junction_Lat': 43.7685461, 'Junction_Lon': -79.3265949}, {'Junction_Lat': 43.7687015, 'Junction_Lon': -79.3140412}, {'Junction_Lat': 43.7678813, 'Junction_Lon': -79.3259759}, {'Junction_Lat': 43.6481414, 'Junction_Lon': -79.3605147}, {'Junction_Lat': 43.6462243, 'Junction_Lon': -79.368164}, {'Junction_Lat': 43.6378351, 'Junction_Lon': -79.3993498}, {'Junction_Lat': 43.6331047, 'Junction_Lon': -79.4283169}, {'Junction_Lat': 43.6341035, 'Junction_Lon': -79.4375266}, {'Junction_Lat': 43.6263761, 'Junction_Lon': -79.4858841}, {'Junction_Lat': 43.6217305, 'Junction_Lon': -79.5050659}, {'Junction_Lat': 43.6203168, 'Junction_Lon': -79.5123354

In [35]:
df_junctions = pd.DataFrame(junctions_list)

In [36]:
query_3 = '''
[out:json][timeout:50];
area["name"="Toronto"]->.searchArea;
(
  node["building"="warehouse"](area.searchArea);
  way["building"="warehouse"](area.searchArea);
  relation["building"="warehouse"](area.searchArea);
);
out center;
'''

In [39]:
response = rq.post(url, data={'data': query_3})

data_dict_3 = response.json()

In [40]:
warehouses_list = []
for element in data_dict_3['elements']:
  if element['type'] == 'node':
    latitude = element['lat']
    longtitude = element['lon']
  elif element['type'] == 'way':
    latitude = element['center']['lat']
    longtitude = element['center']['lon']
  coordinates = {'Warehouse_Lat': latitude, 'Warehouse_Lon': longtitude}
  warehouses_list.append(coordinates)

In [41]:
print(warehouses_list)

[{'Warehouse_Lat': 43.7531556, 'Warehouse_Lon': -79.3562532}, {'Warehouse_Lat': 43.623256, 'Warehouse_Lon': -79.5615121}, {'Warehouse_Lat': 43.626487, 'Warehouse_Lon': -79.5462232}, {'Warehouse_Lat': 43.6191454, 'Warehouse_Lon': -79.5291481}, {'Warehouse_Lat': 43.6079947, 'Warehouse_Lon': -79.5238702}, {'Warehouse_Lat': 43.6284852, 'Warehouse_Lon': -79.5501732}, {'Warehouse_Lat': 43.734017, 'Warehouse_Lon': -79.2892485}, {'Warehouse_Lat': 43.617118, 'Warehouse_Lon': -79.522389}, {'Warehouse_Lat': 43.6067351, 'Warehouse_Lon': -79.5281806}, {'Warehouse_Lat': 43.6031648, 'Warehouse_Lon': -79.5259516}, {'Warehouse_Lat': 43.7154815, 'Warehouse_Lon': -79.5305037}, {'Warehouse_Lat': 43.7013069, 'Warehouse_Lon': -79.600316}, {'Warehouse_Lat': 43.7019038, 'Warehouse_Lon': -79.5979693}, {'Warehouse_Lat': 43.765641, 'Warehouse_Lon': -79.5358492}, {'Warehouse_Lat': 43.6604332, 'Warehouse_Lon': -79.3962749}, {'Warehouse_Lat': 43.6262767, 'Warehouse_Lon': -79.5606519}, {'Warehouse_Lat': 43.6269305, 

In [42]:
df_warehouses = pd.DataFrame(warehouses_list)

In [43]:
query_4 = '''
[out:json][timeout:50];
area["name"="Toronto"]->.searchArea;
(
  node["amenity"="parking"](area.searchArea)(around:4000, 43.6777, -79.6248);
  way["amenity"="parking"](area.searchArea)(around:4000, 43.6777, -79.6248);
  relation["amenity"="parking"](area.searchArea)(around:4000, 43.6777, -79.6248);
);
out center;
'''

In [44]:
response = rq.post(url, data={'data': query_4})

data_dict_4 = response.json()

parking_list = []
for element in data_dict_4['elements']:
  if element['type'] == 'node':
    latitude = element['lat']
    longtitude = element['lon']
  elif element['type'] == 'way':
    latitude = element['center']['lat']
    longtitude = element['center']['lon']
  elif element['type'] == 'relation':
    latitude = element['center']['lat']
    longtitude = element['center']['lon']
  coordinates = {'Parking_Lat': latitude, 'Parking_Lon': longtitude}
  parking_list.append(coordinates)


In [45]:
df_parking = pd.DataFrame(parking_list)
display(df_parking.head())

,Parking_Lat,Parking_Lon
0,43.684253,-79.596358
1,43.659636,-79.580859
2,43.656702,-79.586997
3,43.653703,-79.593381
4,43.678382,-79.588930


In [46]:
query_5 = '''
[out:json][timeout:50];
area["name"="Toronto"]->.searchArea;
(
  node["shop"="car_repair"](area.searchArea);
  way["shop"="car_repair"](area.searchArea);
  relation["shop"="car_repair"](area.searchArea);
);
out center;
'''

In [47]:
response = rq.post(url, data={'data': query_5})

data_dict_5 = response.json()

In [48]:
car_repairments_list = []
for element in data_dict_5['elements']:
  if element['type'] == 'node':
    latitude = element['lat']
    longtitude = element['lon']
  elif element['type'] == 'way':
    latitude = element['center']['lat']
    longtitude = element['center']['lon']
  coordinates = {'Car_Repair_Lat': latitude, 'Car_Repair_Lon': longtitude}
  car_repairments_list.append(coordinates)

In [49]:
df_car_repairments = pd.DataFrame(car_repairments_list)
display(df_car_repairments.head())

,Car_Repair_Lat,Car_Repair_Lon
0,43.691569,-79.303551
1,43.692070,-79.315926
2,43.692219,-79.315385
3,43.679245,-79.295512
4,43.691187,-79.315534


## 3. Spatial Data Transformation (Coordinate Vectorization)

Before constructing the distance matrices, we extract the geographical coordinates (latitude and longitude) across all datasets and convert them from degrees to radians. This vectorization step is required by the Haversine formula to ensure accurate and computationally efficient spherical distance calculations.

In [50]:
import numpy as np
from sklearn.metrics.pairwise import haversine_distances

In [51]:
matrice_autotheft = np.radians(autotheft[['LAT_WGS84', 'LONG_WGS84']].to_numpy())
#print(matrice_autotheft)

In [52]:
matrice_dealers = np.radians(df_dealers[['Dealer_Lat', 'Dealer_Lon']].to_numpy())

In [53]:
matrice_junctions = np.radians(df_junctions[['Junction_Lat', 'Junction_Lon']].to_numpy())

In [54]:
matrice_warehouses = np.radians(df_warehouses[['Warehouse_Lat', 'Warehouse_Lon']].to_numpy())

In [55]:
matrice_parking = np.radians(df_parking[['Parking_Lat', 'Parking_Lon']].to_numpy())


In [56]:
matrice_car_repairments = np.radians(df_car_repairments[['Car_Repair_Lat', 'Car_Repair_Lon']].to_numpy())

## 4. Computing Pairwise Haversine Distances

In this step, we calculate the shortest distance over the earth's surface between every auto theft location and every infrastructure point. The `haversine_distances` function computes the angular distance in radians between points on a sphere. By multiplying this result by `R` (the Earth's mean radius, approximately 6,371 km), we convert the radian values into a physical distance matrix measured in kilometers. This provides a highly accurate spatial relationship between the crimes and the points of interest.

In [57]:
R = 6371

In [91]:
matrice_auto_dealers = haversine_distances(matrice_autotheft, matrice_dealers)*R
matrice_auto_dealers

array([[19.08699111, 21.1142546 ,  9.71159751, ..., 10.35172939,
        11.40650662,  8.21000296],
       [ 4.6505549 ,  1.11509379, 18.30925578, ..., 12.06326578,
        12.21755886, 14.45515253],
       [ 7.14875351,  6.13754059, 23.77768144, ..., 16.95555596,
        17.55036924, 19.3198127 ],
       ...,
       [ 2.67298328,  5.10708299, 14.86649849, ...,  7.49985668,
         8.68786664,  9.75619183],
       [16.46166371, 17.06009578,  1.44591621, ...,  6.44896136,
         5.51293974,  4.60500421],
       [16.46166371, 17.06009578,  1.44591621, ...,  6.44896136,
         5.51293974,  4.60500421]])

## 5. Defining Spatial Thresholds (Radius of Influence)

To quantify whether a theft occurred "near" a specific logistical hub, we must define realistic spatial thresholds. These distances were strategically selected based on the operational logic of auto theft rings and Toronto's urban geography:

* **Highway Junctions (1.0 km):** Represents immediate access. A 1 km distance can be covered in 1-2 minutes, allowing perpetrators to merge onto high-speed escape routes and leave the local jurisdiction before police dispatch.
* **Car Dealerships (1.0 km):** Represents immediate proximity to high-volume vehicle hubs. Dealership lots can serve as direct targets for organized theft rings (e.g., key cloning or relay attacks) or as environments where stolen vehicles temporarily blend in with legitimate inventory.
* **Airport Long-term Parking (1.5 km):** Captures the immediate perimeter of the airport. These massive, largely unmonitored parking structures serve as perfect "cooling off" spots where stolen vehicles can be hidden for days among legitimate travelers' cars.
* **Repair Shops & Dismantlers (2.0 km):** A very short drive that minimizes exposure on public roads. Vehicles can be quickly moved indoors for dismantling ("chop shops"), VIN cloning, or preparing for export.
* **Industrial Warehouses (3.0 km):** Industrial zones are geographically sprawling. A slightly larger 3 km radius accounts for the low-density, low-traffic nature of these districts, providing a discrete environment for criminal logistics and loading containers without keeping the asset too close to the original theft location.

In [62]:
dealers_within_1km = (matrice_auto_dealers <= 1).sum(axis=1)

In [63]:
autotheft['Dealers_in_1km'] = dealers_within_1km
display(autotheft[['NEIGHBOURHOOD_158', 'Dealers_in_1km']].sort_values(by='Dealers_in_1km', ascending=False).head(10))

,NEIGHBOURHOOD_158,Dealers_in_1km
64645,St Lawrence-East Bayfront-The Islands (166),19
8029,Moss Park (73),19
14981,Moss Park (73),19
20100,St Lawrence-East Bayfront-The Islands (166),19
55415,St Lawrence-East Bayfront-The Islands (166),19
18625,St Lawrence-East Bayfront-The Islands (166),19
62040,St Lawrence-East Bayfront-The Islands (166),19
54867,St Lawrence-East Bayfront-The Islands (166),19
7006,Moss Park (73),19
29143,St Lawrence-East Bayfront-The Islands (166),19


In [64]:
top_dealer_thefts = autotheft[autotheft['Dealers_in_1km'] > 0] \
    .groupby('NEIGHBOURHOOD_158') \
    .size() \
    .reset_index(name='Thefts_Near_Dealers') \
    .sort_values(by='Thefts_Near_Dealers', ascending=False)

display(top_dealer_thefts.head(10))

,NEIGHBOURHOOD_158,Thefts_Near_Dealers
118,West Humber-Clairville (1),2647
131,York University Heights (27),1706
40,Etobicoke City Centre (159),1694
132,Yorkdale-Glen Park (31),1114
54,Humber Summit (21),843
29,Dorset Park (126),833
55,Humbermede (22),803
88,Oakdale-Beverley Heights (154),799
123,Wexford/Maryvale (119),735
85,Newtonbrook West (36),733


In [65]:
matrice_auto_junctions = haversine_distances(matrice_autotheft, matrice_junctions)*R


In [66]:
junctions_within_1km = (matrice_auto_junctions <= 1).sum(axis=1)

In [82]:
autotheft['Junctions_in_1km'] = junctions_within_1km
top_junctions_thefts = autotheft[autotheft['Junctions_in_1km'] > 0] \
    .groupby('NEIGHBOURHOOD_158') \
    .size() \
    .reset_index(name='Thefts_Near_Junctions') \
    .sort_values(by='Thefts_Near_Junctions', ascending=False)

display(top_junctions_thefts.head(10))

,NEIGHBOURHOOD_158,Thefts_Near_Junctions
80,West Humber-Clairville (1),3275
24,Etobicoke City Centre (159),1225
57,Oakdale-Beverley Heights (154),980
89,Yorkdale-Glen Park (31),940
14,Clanton Park (33),741
60,Pelmo Park-Humberlea (23),568
16,Don Valley Village (47),535
40,Kingsview Village-The Westway (6),507
6,Bendale-Glen Andrew (156),504
5,Bedford Park-Nortown (39),456


In [67]:
junctions_within_3km = (matrice_auto_junctions <= 1).sum(axis=1)

In [68]:
autotheft['Junctions_in_3km'] = junctions_within_1km
top_junctions_thefts = autotheft[autotheft['Junctions_in_3km'] > 0] \
    .groupby('NEIGHBOURHOOD_158') \
    .size() \
    .reset_index(name='Thefts_Near_Junctions') \
    .sort_values(by='Thefts_Near_Junctions', ascending=False)

display(top_junctions_thefts.head(10))

,NEIGHBOURHOOD_158,Thefts_Near_Junctions
80,West Humber-Clairville (1),3275
24,Etobicoke City Centre (159),1225
57,Oakdale-Beverley Heights (154),980
89,Yorkdale-Glen Park (31),940
14,Clanton Park (33),741
60,Pelmo Park-Humberlea (23),568
16,Don Valley Village (47),535
40,Kingsview Village-The Westway (6),507
6,Bendale-Glen Andrew (156),504
5,Bedford Park-Nortown (39),456


In [69]:
matrice_auto_warehouses = haversine_distances(matrice_autotheft, matrice_warehouses)*R

In [70]:
autotheft['Warehouses_in_3km'] = (matrice_auto_warehouses <= 3).sum(axis=1)

top_warehouses = autotheft[autotheft['Warehouses_in_3km'] > 0] \
    .groupby('NEIGHBOURHOOD_158').size() \
    .reset_index(name='Thefts_Near_Warehouses') \
    .sort_values(by='Thefts_Near_Warehouses', ascending=False)
display(top_warehouses.head(10))

,NEIGHBOURHOOD_158,Thefts_Near_Warehouses
72,West Humber-Clairville (1),4088
25,Etobicoke City Centre (159),1724
35,Humber Summit (21),1111
55,Oakdale-Beverley Heights (154),973
36,Humbermede (22),844
75,Wexford/Maryvale (119),839
30,Glenfield-Jane Heights (25),755
74,Weston (113),753
10,Black Creek (24),750
58,Pelmo Park-Humberlea (23),713


In [71]:
matrice_auto_repair = haversine_distances(matrice_autotheft, matrice_car_repairments)*R

In [72]:
autotheft['RepairShops_in_2km'] = (matrice_auto_repair <= 2.0).sum(axis=1)

top_repair = autotheft[autotheft['RepairShops_in_2km'] > 0] \
    .groupby('NEIGHBOURHOOD_158').size() \
    .reset_index(name='Thefts_Near_RepairShops') \
    .sort_values(by='Thefts_Near_RepairShops', ascending=False)

display(top_repair.head(5))

,NEIGHBOURHOOD_158,Thefts_Near_RepairShops
139,West Humber-Clairville (1),5002
156,York University Heights (27),1970
46,Etobicoke City Centre (159),1724
145,Wexford/Maryvale (119),1256
65,Humber Summit (21),1248


In [73]:
matrice_auto_parking = haversine_distances(matrice_autotheft, matrice_parking) * 6371

autotheft['AirportParking_in_1_5km'] = (matrice_auto_parking <= 1.5).sum(axis=1)

top_airport = autotheft[autotheft['AirportParking_in_1_5km'] > 0] \
    .groupby('NEIGHBOURHOOD_158').size() \
    .reset_index(name='Thefts_Near_Airport') \
    .sort_values(by='Thefts_Near_Airport', ascending=False)

display(top_airport.head(5))

,NEIGHBOURHOOD_158,Thefts_Near_Airport
6,West Humber-Clairville (1),3269
0,Eringate-Centennial-West Deane (11),624
7,Willowridge-Martingrove-Richview (7),382
1,Etobicoke West Mall (13),91
3,Kingsview Village-The Westway (6),63


## 6. Statistical Hypothesis Testing: Highway Proximity

To determine if the observed concentration of thefts near highways in West Humber is statistically significant, we apply t-test. This test is appropriate for comparing the means of two independent samples with unequal variances and different sample sizes.

**Null Hypothesis ($H_0$):** There is no difference in the proportion of auto thefts occurring near highways (<= 1 km) between West Humber and the rest of Toronto.
**Alternative Hypothesis ($H_1$):** A significantly higher proportion of auto thefts occur near highways in West Humber compared to the rest of the city.

In [100]:
from scipy.stats import ttest_ind

In [101]:
autotheft['is_near_highway'] = (autotheft['Junctions_in_1km'] > 0).astype(int)

wh_array = autotheft[autotheft['NEIGHBOURHOOD_158'] == 'West Humber-Clairville (1)']['is_near_highway']
other_array = autotheft[autotheft['NEIGHBOURHOOD_158'] != 'West Humber-Clairville (1)']['is_near_highway']


In [102]:
t_stat, p_value = ttest_ind(wh_array, other_array, equal_var=False)

In [103]:
print(f"West Humber Mean Proportion:  {wh_array.mean():.1%}")
print(f"Rest of Toronto Mean Proportion: {other_array.mean():.1%}")
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.3e}\n")

if p_value < 0.05:
    print("Conclusion: Reject the null hypothesis.")
    print("Interpretation: Auto thefts in West Humber occur near highways at a statistically significantly higher rate.")
else:
    print("Conclusion: Fail to reject the null hypothesis.")
    print("Interpretation: No statistically significant difference observed.")

West Humber Mean Proportion:  55.7%
Rest of Toronto Mean Proportion: 30.7%
T-statistic: 37.325
P-value: 2.296e-277

Conclusion: Reject the null hypothesis.
Interpretation: Auto thefts in West Humber occur near highways at a statistically significantly higher rate.


## 7. Statistical Hypothesis Testing: Car Dealerships Proximity.

In [104]:
autotheft['is_near_dealer'] = (autotheft['Dealers_in_500m'] > 0).astype(int)

In [105]:
wh_dealers_array = autotheft[autotheft['NEIGHBOURHOOD_158'] == 'West Humber-Clairville (1)']['is_near_dealer']
other_dealers_array = autotheft[autotheft['NEIGHBOURHOOD_158'] != 'West Humber-Clairville (1)']['is_near_dealer']

In [106]:
t_stat_d, p_value_d = ttest_ind(wh_dealers_array, other_dealers_array, equal_var=False)

In [108]:
print("--- CAR DEALERSHIPS (500 m) ---")
print(f"West Humber Mean Proportion:  {wh_dealers_array.mean():.1%}")
print(f"Rest of Toronto Mean Proportion: {other_dealers_array.mean():.1%}")
print(f"T-statistic: {t_stat_d:.3f}")
print(f"P-value: {p_value_d:.3e}\n")

if p_value_d < 0.05:
    print("Conclusion: Reject the null hypothesis.")
    print("Interpretation: Auto thefts in West Humber occur near car dealerships at a statistically significantly higher rate.")
else:
    print("Conclusion: Fail to reject the null hypothesis.")
    print("Interpretation: No statistically significant difference observed. Proximity to dealerships in West Humber is consistent with the city average.")

--- CAR DEALERSHIPS (500 m) ---
West Humber Mean Proportion:  19.6%
Rest of Toronto Mean Proportion: 20.3%
T-statistic: -1.333
P-value: 1.824e-01

Conclusion: Fail to reject the null hypothesis.
Interpretation: No statistically significant difference observed. Proximity to dealerships in West Humber is consistent with the city average.


## 8. Statistical Hypothesis Testing: Industrial Warehouses Proximity.

In [109]:
autotheft['is_near_warehouse'] = (autotheft['Warehouses_in_3km'] > 0).astype(int)

In [110]:
wh_warehouses_array = autotheft[autotheft['NEIGHBOURHOOD_158'] == 'West Humber-Clairville (1)']['is_near_warehouse']
other_warehouses_array = autotheft[autotheft['NEIGHBOURHOOD_158'] != 'West Humber-Clairville (1)']['is_near_warehouse']

In [111]:
t_stat_w, p_value_w = ttest_ind(wh_warehouses_array, other_warehouses_array, equal_var=False)

In [113]:
print("--- INDUSTRIAL WAREHOUSES (3.0 km) ---")
print(f"West Humber Mean Proportion:  {wh_warehouses_array.mean():.1%}")
print(f"Rest of Toronto Mean Proportion: {other_warehouses_array.mean():.1%}")
print(f"T-statistic: {t_stat_w:.3f}")
print(f"P-value: {p_value_w:.3e}\n")

if p_value_w < 0.05:
    print("Conclusion: Reject the null hypothesis.")
    print("Interpretation: Auto thefts in West Humber occur near industrial warehouses at a statistically significantly higher rate.")
else:
    print("Conclusion: Fail to reject the null hypothesis.")
    print("Interpretation: No statistically significant difference observed.")

--- INDUSTRIAL WAREHOUSES (3.0 km) ---
West Humber Mean Proportion:  69.5%
Rest of Toronto Mean Proportion: 36.8%
T-statistic: 52.260
P-value: 0.000e+00

Conclusion: Reject the null hypothesis.
Interpretation: Auto thefts in West Humber occur near industrial warehouses at a statistically significantly higher rate.


## 9. Statistical Hypothesis Testing: Airport Parking Proximity.

In [114]:
autotheft['is_near_airport_parking'] = (autotheft['AirportParking_in_1_5km'] > 0).astype(int)

In [115]:
wh_airport = autotheft[autotheft['NEIGHBOURHOOD_158'] == 'West Humber-Clairville (1)']['is_near_airport_parking']
other_airport = autotheft[autotheft['NEIGHBOURHOOD_158'] != 'West Humber-Clairville (1)']['is_near_airport_parking']

In [116]:
t_stat_a, p_value_a = ttest_ind(wh_airport, other_airport, equal_var=False)

In [118]:
print("--- AIRPORT PARKING (1.5 km) ---")
print(f"West Humber Mean Proportion:  {wh_airport.mean():.1%}")
print(f"Rest of Toronto Mean Proportion: {other_airport.mean():.1%}")
print(f"T-statistic: {t_stat_a:.3f}")
print(f"P-value: {p_value_a:.3e}\n")

if p_value_a < 0.05:
    print("Conclusion: Reject the null hypothesis.")
    print("Interpretation: Auto thefts in West Humber occur near airport parking at a statistically significantly higher rate.")
else:
    print("Conclusion: Fail to reject the null hypothesis.")
    print("Interpretation: No statistically significant difference observed")

--- AIRPORT PARKING (1.5 km) ---
West Humber Mean Proportion:  55.6%
Rest of Toronto Mean Proportion: 1.7%
T-statistic: 82.944
P-value: 0.000e+00

Conclusion: Reject the null hypothesis.
Interpretation: Auto thefts in West Humber occur near airport parking at a statistically significantly higher rate.


## 10. Statistical Hypothesis Testing: Repair Shops Proximity.

In [119]:
autotheft['is_near_repair_shop'] = (autotheft['RepairShops_in_2km'] > 0).astype(int)

In [120]:
wh_repair = autotheft[autotheft['NEIGHBOURHOOD_158'] == 'West Humber-Clairville (1)']['is_near_repair_shop']
other_repair = autotheft[autotheft['NEIGHBOURHOOD_158'] != 'West Humber-Clairville (1)']['is_near_repair_shop']

In [121]:
t_stat_r, p_value_r = ttest_ind(wh_repair, other_repair, equal_var=False)

In [122]:
print("--- AUTO REPAIR SHOPS (2.0 km) ---")
print(f"West Humber Mean Proportion:  {wh_repair.mean():.1%}")
print(f"Rest of Toronto Mean Proportion: {other_repair.mean():.1%}")
print(f"T-statistic: {t_stat_r:.3f}")
print(f"P-value: {p_value_r:.3e}\n")

if p_value_r < 0.05:
    print("Conclusion: Reject the null hypothesis.")
    print("Interpretation: A statistically significant difference is observed. However, the proportion in West Humber is actually lower than the city average, indicating this is not the unique driver for the neighborhood's theft spike.")
else:
    print("Conclusion: Fail to reject the null hypothesis.")
    print("Interpretation: No statistically significant difference observed.")

--- AUTO REPAIR SHOPS (2.0 km) ---
West Humber Mean Proportion:  85.1%
Rest of Toronto Mean Proportion: 90.3%
T-statistic: -10.999
P-value: 6.765e-28

Conclusion: Reject the null hypothesis.
Interpretation: A statistically significant difference is observed. However, the proportion in West Humber is actually lower than the city average, indicating this is not the unique driver for the neighborhood's theft spike.
